**6.3 Encoding Categorical Variables**

Steps:
1. Load train and test splits
2. Find categorical columns
3. Apply LabelEncoding for ordinal categories
4. Apply OneHotEncoding for nominal categories
5. Save encoded train and test sets

---

### Label Encoding

Turns each category into a number. Use for **ordinal** or **binary** categories.

**Example:**

```
Size        Size_Encoded
Small       0
Medium      1
Large       2
```

### One Hot Encoding

Creates a new column for each category. Use for **nominal** categories.

**Example:**

```
Color    Color_Red    Color_Blue    Color_Green
Red      1            0             0
Blue     0            1             0
Green    0            0             1
```

**Summary:**
Label encoding for ordered categories. One hot encoding for unordered categories.

---


**1. Setup**

In [1]:
%pip install -q pandas numpy scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

pd.set_option("display.max_columns", None)

**2. Load the imputed data**

In [3]:
train_df = pd.read_csv("titanic_train_imputed.csv")
test_df = pd.read_csv("titanic_test_imputed.csv")

target = "survived"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("Train X shape:", X_train.shape)
print("Test X shape :", X_test.shape)

Train X shape: (712, 13)
Test X shape : (179, 13)


In [4]:
X_train.head()

,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alone
0,3,male,28.5,0,0,56.4958,S,Third,man,True,C,Southampton,True
1,2,male,28.5,0,0,0.0000,S,Second,man,True,C,Southampton,True
2,1,male,28.5,0,0,221.7792,S,First,man,True,C,Southampton,True
3,3,female,18.0,0,1,9.3500,S,Third,woman,False,C,Southampton,False
4,2,female,31.0,1,1,26.2500,S,Second,woman,False,C,Southampton,False


In [5]:
X_train["embark_town"].value_counts()

embark_town
Southampton    518
Cherbourg      139
Queenstown      55
Name: count, dtype: int64

In [6]:
X_train["embarked"].value_counts()

embarked
S    518
C    139
Q     55
Name: count, dtype: int64

In [7]:
y_train.head()

0    1
1    0
2    0
3    1
4    1
Name: survived, dtype: int64

In [8]:
# remove redundant columns - class and embark town
X_train = X_train.drop(columns=["class", "embark_town"])
X_test = X_test.drop(columns=["class", "embark_town"])

In [9]:
X_train.head()

,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,3,male,28.5,0,0,56.4958,S,man,True,C,True
1,2,male,28.5,0,0,0.0000,S,man,True,C,True
2,1,male,28.5,0,0,221.7792,S,man,True,C,True
3,3,female,18.0,0,1,9.3500,S,woman,False,C,False
4,2,female,31.0,1,1,26.2500,S,woman,False,C,False


In [10]:
X_test.head()

,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,3,male,24.0,2,0,24.1500,S,man,True,C,False
1,3,male,44.0,0,1,16.1000,S,man,True,C,False
2,3,male,22.0,0,0,7.2250,C,man,True,C,True
3,3,male,41.0,2,0,14.1083,S,man,True,C,False
4,3,female,28.5,1,0,15.5000,Q,woman,False,C,False


**3. Identify categorical and numeric columns**

In [11]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ['pclass', 'sex', 'embarked', 'who', 'adult_male', 'deck', 'alone']

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['age', 'sibsp', 'parch', 'fare']
Categorical columns: ['pclass', 'sex', 'embarked', 'who', 'adult_male', 'deck', 'alone']


**4. Split into binary, ordinal categories (label encoding) & nominal categories (one hot encoding)**

In [12]:
X_train.head()

,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,3,male,28.5,0,0,56.4958,S,man,True,C,True
1,2,male,28.5,0,0,0.0000,S,man,True,C,True
2,1,male,28.5,0,0,221.7792,S,man,True,C,True
3,3,female,18.0,0,1,9.3500,S,woman,False,C,False
4,2,female,31.0,1,1,26.2500,S,woman,False,C,False


In [19]:
for col in cat_cols:
    print(f"{col}: {X_train[col].unique()}")
    print("-" * 30)

pclass: [3 2 1]
------------------------------
sex: <StringArray>
['male', 'female']
Length: 2, dtype: str
------------------------------
embarked: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: str
------------------------------
who: <StringArray>
['man', 'woman', 'child']
Length: 3, dtype: str
------------------------------
adult_male: [ True False]
------------------------------
deck: <StringArray>
['C', 'A', 'D', 'E', 'F', 'B', 'G']
Length: 7, dtype: str
------------------------------
alone: [ True False]
------------------------------


| Column       | Type     | Explanation |
|--------------|----------|-------------|
| **pclass**       | Ordinal | Travel classes have a natural order (1 > 2 > 3) |
| **sex**          | Nominal (binary) | No ranking between male and female |
| **embarked**     | Nominal | Ports do not have any order |
| **who**          | Nominal *(optional ordinal)* | Usually treated as nominal unless modeling age groups |
| **adult_male**   | Nominal (binary) | Binary indicator, no inherent order |
| **deck**         | Nominal | Cabin letters do not represent a meaningful order |
| **alone**        | Nominal (binary) | Binary indicator, no hierarchy |

In [20]:
binary_and_ordinal_cols = ["sex", "adult_male", "alone"]  # label enc
multi_class_nominal_cols = ["embarked", "who", "deck"]  # one hot enc

In [21]:
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

**5. Label Encoding**

In [22]:
binary_and_ordinal_cols

['sex', 'adult_male', 'alone']

In [29]:
label_encoders = {} # Creates an empty dictionary to store one LabelEncoder per column.

for col in binary_and_ordinal_cols:
    le = LabelEncoder() # Creates a fresh encoder for the current column.
    le.fit(X_train[col]) # Learns the unique categories in that column from the training data.
    label_encoders[col] = le # Stores the fitted encoder so you can use it later on test data or new data.
    X_train_enc[col] = le.transform(X_train[col]) # Replaces the original text categories in the training set with numbers.
    X_test_enc[col] = le.transform(X_test[col]) # Replaces the original text categories in the test set with numbers.

In [30]:
label_encoders

{'sex': LabelEncoder(), 'adult_male': LabelEncoder(), 'alone': LabelEncoder()}

In [31]:
X_train.head()

,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,3,male,28.5,0,0,56.4958,S,man,True,C,True
1,2,male,28.5,0,0,0.0000,S,man,True,C,True
2,1,male,28.5,0,0,221.7792,S,man,True,C,True
3,3,female,18.0,0,1,9.3500,S,woman,False,C,False
4,2,female,31.0,1,1,26.2500,S,woman,False,C,False


In [28]:
X_train_enc.head()

,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,deck,alone
0,3,1,28.5,0,0,56.4958,S,man,1,C,1
1,2,1,28.5,0,0,0.0000,S,man,1,C,1
2,1,1,28.5,0,0,221.7792,S,man,1,C,1
3,3,0,18.0,0,1,9.3500,S,woman,0,C,0
4,2,0,31.0,1,1,26.2500,S,woman,0,C,0


**IMPORTANT: The encoders will be saved for later prediction use on new data**

**7. One Hot Encoding**

**Dummy variable trap**

**Dimensional Complexity**

In [35]:
ohe = OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False) 
# drop="first" to avoid dummy variable trap, handle_unknown="ignore" to ignore unseen 
# categories in test data, sparse_output=False to get a dense array output

**Why we use `OneHotEncoder` here**

- `fit()` learns the unique categories from the training data for the selected nominal columns.
- `transform()` converts each category into a binary vector using the same learned categories.
- `drop="first"` removes the first category to avoid the dummy variable trap.
- `handle_unknown="ignore"` prevents errors if the test set contains a category not seen during training.
- `sparse_output=False` returns a normal dense NumPy array instead of a sparse matrix.

**Tiny dry run:**
If `embarked = ["S", "C", "Q"]` in training, the encoder learns those categories and may drop `"C"`. Then:
- `C -> [0, 0]`
- `Q -> [1, 0]`
- `S -> [0, 1]`

The same mapping is then reused for test data so train and test stay consistent.

In [39]:
ohe.fit(X_train[multi_class_nominal_cols]) # fit the encoder on the training data's multi-class nominal columns

,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infre

In [34]:
train_ohe = ohe.transform(X_train[multi_class_nominal_cols])
test_ohe = ohe.transform(X_test[multi_class_nominal_cols])

In [36]:
train_ohe

array([[0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 1., 1., ..., 1., 0., 0.],
       [0., 1., 1., ..., 0., 0., 0.]], shape=(712, 10))

In [38]:
# ensure encoder is fitted before requesting feature names (handles out-of-order cell execution)
if not hasattr(ohe, "categories_"):
    ohe.fit(X_train[multi_class_nominal_cols])
ohe_cols = ohe.get_feature_names_out(multi_class_nominal_cols)

print("oneHotEncoder columns:", ohe_cols)

oneHotEncoder columns: ['embarked_Q' 'embarked_S' 'who_man' 'who_woman' 'deck_B' 'deck_C'
 'deck_D' 'deck_E' 'deck_F' 'deck_G']


In [42]:
train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=X_train.index)
# create a DataFrame from the one-hot encoded array, using the generated column names and original index
test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=X_test.index) 
# create a DataFrame from the one-hot encoded array, using the generated column names and original index

In [41]:
train_ohe_df.head()

,embarked_Q,embarked_S,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G
0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


**7. Remove original multi-class nominal columns and join teh encoded**

In [43]:
multi_class_nominal_cols

['embarked', 'who', 'deck']

In [46]:
X_train_enc = X_train_enc.drop(columns=multi_class_nominal_cols, errors="ignore")

if not set(multi_class_nominal_cols).issubset(X_test_enc.columns):
    X_test_enc = X_test_enc.join(X_test[multi_class_nominal_cols])
X_test_enc = X_test_enc.drop(columns=multi_class_nominal_cols)

In [47]:
print(X_train_enc.shape)
X_train_enc.head()

(712, 8)


,pclass,sex,age,sibsp,parch,fare,adult_male,alone
0,3,1,28.5,0,0,56.4958,1,1
1,2,1,28.5,0,0,0.0000,1,1
2,1,1,28.5,0,0,221.7792,1,1
3,3,0,18.0,0,1,9.3500,0,0
4,2,0,31.0,1,1,26.2500,0,0


In [48]:
print(train_ohe_df.shape)
train_ohe_df.head()

(712, 10)


,embarked_Q,embarked_S,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G
0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


In [49]:
X_train_enc = pd.concat([X_train_enc, train_ohe_df], axis=1)
X_test_enc = pd.concat([X_test_enc, test_ohe_df], axis=1)

In [50]:
X_train_enc.head()

,pclass,sex,age,sibsp,parch,fare,adult_male,alone,embarked_Q,embarked_S,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G
0,3,1,28.5,0,0,56.4958,1,1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2,1,28.5,0,0,0.0000,1,1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,1,1,28.5,0,0,221.7792,1,1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,3,0,18.0,0,1,9.3500,0,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,2,0,31.0,1,1,26.2500,0,0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


In [51]:
X_test_enc.head()

,pclass,sex,age,sibsp,parch,fare,adult_male,alone,embarked_Q,embarked_S,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G
0,3,1,24.0,2,0,24.1500,1,0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,3,1,44.0,0,1,16.1000,1,0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,1,22.0,0,0,7.2250,1,1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,3,1,41.0,2,0,14.1083,1,0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,3,0,28.5,1,0,15.5000,0,0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


**Note: # ML model understands only numerical data, so all features must be numeric after encoding., we dont have any object here, all are int64 or float64**

In [54]:
X_train_enc.dtypes 

pclass          int64
sex             int64
age           float64
sibsp           int64
parch           int64
fare          float64
adult_male      int64
alone           int64
embarked_Q    float64
embarked_S    float64
who_man       float64
who_woman     float64
deck_B        float64
deck_C        float64
deck_D        float64
deck_E        float64
deck_F        float64
deck_G        float64
dtype: object

In [57]:
print("Shapes after encoding:")
print("X_train_enc:", X_train_enc.shape)
print("X_test_enc :", X_test_enc.shape)

# if no of categorical variables are large, then we can use target encoding or frequency encoding instead of one hot encoding to avoid curse of dimensionality.

Shapes after encoding:
X_train_enc: (712, 18)
X_test_enc : (179, 18)


**9. Save encoded sets**

In [56]:
train_enc_df = X_train_enc.copy()
train_enc_df[target] = y_train.values

test_enc_df = X_test_enc.copy()
test_enc_df[target] = y_test.values

train_enc_df.to_csv("titanic_train_encoded.csv", index=False)
test_enc_df.to_csv("titanic_test_encoded.csv", index=False)

print("\nSaved:")
print(" - titanic_train_encoded.csv")
print(" - titanic_test_encoded.csv")


Saved:
 - titanic_train_encoded.csv
 - titanic_test_encoded.csv
